In [1]:
from dotenv import load_dotenv
from langchain_openai import AzureChatOpenAI, AzureOpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore
import os

In [2]:
load_dotenv()

True

In [3]:
#langchain tracking
os.environ["LANGSMITH_API_KEY"] = os.getenv("LANGSMITH_API_KEY")
os.environ["LANGSMITH_PROJECT"] = os.getenv("LANGSMITH_PROJECT")
os.environ["LANGSMITH_PROJECT"] = os.getenv("LANGSMITH_PROJECT")
os.environ["LANGSMITH_ENDPOINT"] = os.getenv("LANGSMITH_ENDPOINT")

In [4]:
from langchain_community.document_loaders import WebBaseLoader

USER_AGENT environment variable not set, consider setting it to identify your requests.


##### Loading Web site pages

In [5]:
loader = WebBaseLoader("https://tenscit.com/")
docs = loader.load()

In [6]:
docs

[Document(metadata={'source': 'https://tenscit.com/', 'title': ' Tenscit - Advanced Writing Analysis Tool', 'description': 'Professional-grade AI writing analysis for students and researchers. Grammar analysis, citation generation, paraphrasing, and improve your academic writing with our comprehensive suite of tools.', 'language': 'en'}, page_content=" Tenscit - Advanced Writing Analysis ToolTENSCITLoading your experience...FeaturesGrammarCitationParaphraseAI TranslatorAI DetectorResearch Paper AnalyserOriginality CheckerPricingSupportSign InGet Started☰Write like a proMaster Your Research With The All-in-One AI Academic Writing AssistantExpertly format APA, MLA, and Chicago citations while polishing your academic tone with AI-powered precision. Designed for scholars.Start Writing SmarterSee Pricing Key FeaturesAI Powered Tools Built for Academic ExcellenceTools meticulously crafted to meet the rigorous standards of modern research.APA/MLA Citation BuilderAutomated formatting for any s

##### Chunking

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [8]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = text_splitter.split_documents(docs)

In [9]:
chunks

[Document(metadata={'source': 'https://tenscit.com/', 'title': ' Tenscit - Advanced Writing Analysis Tool', 'description': 'Professional-grade AI writing analysis for students and researchers. Grammar analysis, citation generation, paraphrasing, and improve your academic writing with our comprehensive suite of tools.', 'language': 'en'}, page_content='Tenscit - Advanced Writing Analysis ToolTENSCITLoading your experience...FeaturesGrammarCitationParaphraseAI TranslatorAI DetectorResearch Paper AnalyserOriginality CheckerPricingSupportSign InGet Started☰Write like a proMaster Your Research With The All-in-One AI Academic Writing AssistantExpertly format APA, MLA, and Chicago citations while polishing your academic tone with AI-powered precision. Designed for scholars.Start Writing SmarterSee Pricing Key FeaturesAI Powered Tools Built for'),
 Document(metadata={'source': 'https://tenscit.com/', 'title': ' Tenscit - Advanced Writing Analysis Tool', 'description': 'Professional-grade AI wr

##### Embeddings

In [10]:
embeddings = AzureOpenAIEmbeddings(
    azure_deployment=os.environ.get("AZURE_OPENAI_EMBEDDING_DEPLOYMENT_NAME"),
    azure_endpoint=os.environ.get("AZURE_OPENAI_ENDPOINT"),
    api_key=os.environ.get("AZURE_OPENAI_API_KEY"),
    api_version=os.environ.get("AZURE_OPENAI_API_VERSION")
)

##### Embeddings into vectore store

In [11]:
vector_store = PineconeVectorStore.from_documents(
    documents=chunks,
    embedding=embeddings,
    index_name=os.environ.get("PINECONE_INDEx_NAME"))

In [12]:
llm = AzureChatOpenAI(
    api_key = os.getenv("AZURE_OPENAI_API_KEY"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
    azure_deployment = os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME"),
    azure_endpoint= os.getenv("AZURE_OPENAI_ENDPOINT"),
)

##### Query from vector store

In [13]:
query = "Work meets the highest"
result = vector_store.similarity_search(query)

In [14]:
result[0].page_content

'work meets the highest standards.Upload Your DocumentPaste or upload your text securely for quick academic analysis.AI ProcessingAI scans your content for structure, grammar, citations, and clarity.Review & ImproveGet clear feedback to refine accuracy and overall writing quality.After TenscitFrom Rough Drafts to Published PapersQarin brings your team, clients, and data into one powerful workspace, turning everyday clutter into a smooth, focused rhythm of growth.200+ teams trust TenscitJoin'

In [15]:
#Retrival chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template(
    """ 
Answer the following questions based on the following context
<context>
{context}
<context>

"""
)

In [16]:
chain_document = create_stuff_documents_chain(llm,prompt)
chain_document

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template=' \nAnswer the following questions based on the following context\n<context>\n{context}\n<context>\n\n'), additional_kwargs={})])
| AzureChatOpenAI(profile={'name': 'GPT-5.1', 'release_date': '2025-11-13', 'last_updated': '2025-11-13', 'open_weights': False, 'max_input_tokens': 272000, 'max_output_tokens': 128000, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 't

In [17]:
from langchain_core.documents import Document
chain_document.invoke({
    "input": "Work meets the highest",
    "context": [Document(page_content="work meets the highest standards.Upload Your DocumentPaste or upload your text securely for quick academic analysis.AI ProcessingAI scans your content for structure, grammar, citations, and clarity.Review & ImproveGet clear feedback to refine accuracy and overall writing quality.")]

})

'1. **What is the main purpose described in the context?**  \nTo provide a service that analyzes academic writing and helps improve its quality.\n\n2. **What is the first step in the described process?**  \nUpload or paste your document securely into the system.\n\n3. **What does the AI do after the document is uploaded?**  \nIt scans the content for structure, grammar, citations, and clarity.\n\n4. **What is the final step for the user?**  \nReview the feedback and use it to refine the accuracy and overall quality of the writing.\n\n5. **How is security addressed in the context?**  \nIt mentions that you can paste or upload your text “securely,” implying protection of the document during upload.\n\n6. **What aspects of writing quality are specifically mentioned?**  \nStructure, grammar, citations, clarity, accuracy, and overall writing quality.'

In [21]:
retriever = vector_store.as_retriever()

from langchain_classic.chains import create_retrieval_chain

In [22]:
retriever_chain = create_retrieval_chain(retriever, chain_document)
retriever_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['PineconeVectorStore', 'AzureOpenAIEmbeddings'], vectorstore=<langchain_pinecone.vectorstores.PineconeVectorStore object at 0x000002CF349F6DD0>, search_kwargs={}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | ChatPromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template=' \nAnswer the following questions based on the following context\n<context>\n{context}\n<context>\n\n'), additional_kwargs=

In [26]:
response = retriever_chain.invoke({"input": "Work meets the highest"})
response["answer"]

'Go ahead and send the questions you’d like answered.  \n\nI’ve read the provided context and will base my answers strictly on that text, plus general reasoning where appropriate.'